# STITCHV2 Whole-Genome Run and Pedigree QC

This notebook is a server-side runbook for STITCHV2 across all chromosomes. It covers input validation, chromosome-specific ploidy settings, immutable founders plus one mutable founder, compact evidence caching, first-pass no-pedigree runs, global pedigree QC, and second-pass runs with the curated pedigree.

The default behavior in this notebook is `DRY_RUN = True`, which prints commands instead of executing them. Review the commands and set `DRY_RUN = False` only when the paths and cluster settings are correct.

## Workflow

1. Validate the single conda environment and the installed `stitchv2` entrypoint.
2. Validate input tables: samples, positions, founders, and pedigree.
3. Run STITCHV2 once per chromosome without pedigree. This first pass creates genotype calls for pedigree QC and can also write/read compact evidence caches.
4. Run `stitchv2 pedigree-qc` using preliminary genotype calls from all chromosomes.
5. Review `pedigree_qc_summary.json`, parent-edge QC, unexpected relatedness pairs, and the UMAP before/after edge plot.
6. Rerun each chromosome with `pedigree_curated.parquet` using `transmission`, `kinship`, or `smooth`.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import shlex
import subprocess
import sys

import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# Edit these paths for the server run.
REPO = Path('/home/bonnie/Documents/codex/STITCHV2')
STITCHV2 = os.environ.get('STITCHV2_BIN', 'stitchv2')

DATA_DIR = Path('/path/to/full_dataset')
SAMPLES = DATA_DIR / 'samples.parquet'
POSITIONS = DATA_DIR / 'positions.parquet'
PEDIGREE = DATA_DIR / 'pedigree.parquet'

# Founder PLINK prefix, not an individual .bed/.bim/.fam file.
# STITCHV2 strips a leading 'chr' for PLINK chromosome matching, so --chromosome chr12 can match PLINK chrom 12.
FOUNDER_PLINK = DATA_DIR / 'founders8'

RUN_ROOT = Path('/path/to/stitchv2_runs')
CACHE_ROOT = RUN_ROOT / 'compact_evidence_cache'
JAX_CACHE = RUN_ROOT / 'jax_compile_cache'
PEDIGREE_QC_DIR = RUN_ROOT / 'pedigree_qc_global'

# Keep this True until every printed command looks correct.
DRY_RUN = True

# Common model/run settings.
N_FOUNDERS = 9          # 8 immutable founders from PLINK + 1 extra mutable founder.
EM_ITERATIONS = 40      # Enough for mutable-founder convergence; adaptive EM may stop earlier.
MAX_MEM = '90%'         # Use OS memory-aware block/window planning.
HMM_BACKEND = 'jax'
READ_BACKEND = 'stitch_style_bamreader'
GENOTYPE_CALL_MODE = 'stitch_no_call'
CALIBRATION_CALLABILITY_DECISION_MODE = 'per_snp_hierarchical'

# Dask local defaults: thread workers avoid serialization overhead and are more stable on one node.
# If you launch chromosomes in parallel jobs, set DASK_DASHBOARD_ADDRESS='' or a unique port per job.
EXECUTOR = 'dask'
DASK_SCHEDULER = 'local'
DASK_N_WORKERS = 0
DASK_THREADS_PER_WORKER = 1
DASK_MEMORY_LIMIT = ''
DASK_DASHBOARD_ADDRESS = ':8787'

RUN_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
JAX_CACHE.mkdir(parents=True, exist_ok=True)

print('stitchv2:', STITCHV2)
print('run root:', RUN_ROOT)


## Environment Validation

Use one conda environment for the full benchmark or production run. Install the package into that environment with `pip install -e .` so the CLI entrypoint is `stitchv2`, not `python -c ...`.

In [ ]:
def run_checked(cmd: list[str], *, cwd: Path | None = None, dry_run: bool | None = None) -> subprocess.CompletedProcess | None:
    text = ' '.join(shlex.quote(str(x)) for x in cmd)
    print('\n$ ' + text)
    do_dry_run = DRY_RUN if dry_run is None else dry_run
    if do_dry_run:
        return None
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True, text=True)

def capture(cmd: list[str]) -> str:
    return subprocess.check_output(cmd, text=True, stderr=subprocess.STDOUT)

# These should all succeed before the whole-genome run.
for cmd in ([STITCHV2, '--help'], [STITCHV2, 'run', '--help'], [STITCHV2, 'pedigree-qc', '--help'], [STITCHV2, 'combine', '--help']):
    try:
        out = capture(cmd)
        print('OK:', ' '.join(cmd), 'first line:', out.splitlines()[0])
    except Exception as exc:
        print('FAILED:', ' '.join(cmd), exc)
        raise

import stitchv2
print('Python:', sys.executable)
print('stitchv2 package:', stitchv2.__file__)


## Input Formats

`samples.parquet` or CSV must contain:

| column | required | meaning |
| --- | --- | --- |
| `sample_id` | yes | unique STITCHV2 sample id; also used by pedigree QC |
| `bam_path` | yes | BAM/CRAM path; indexes must exist next to each file |
| `generation` | recommended | sample generation used by STITCH-like recombination settings |
| `sex` | required for sex-aware ploidy | accepted values include `M`, `F`, `XY`, `XX` |
| `father_id`, `mother_id` | optional | can be in samples if no separate pedigree is supplied |

`positions.parquet` or CSV must contain `CHR`, `POS`, `REF`, `ALT`. For BAM reading, `CHR` and `--chromosome` should match BAM contig names, for example `chr12`. PLINK founders can still use numeric `12` because the PLINK loader strips a leading `chr`.

`pedigree.parquet` or CSV must contain an offspring column and at least one parent column. Defaults are `sample_id`, `father_id`, `mother_id`. Missing parents can be empty, `NA`, `NaN`, `none`, `null`, or `0`.

In [ ]:
def read_table(path: Path) -> pd.DataFrame:
    if path.suffix == '.parquet':
        return pd.read_parquet(path)
    return pd.read_csv(path, sep=None, engine='python')

def validate_inputs() -> None:
    for path in [SAMPLES, POSITIONS, PEDIGREE]:
        print(path, 'exists:', path.exists())
    for suffix in ['.bed', '.bim', '.fam']:
        print(str(FOUNDER_PLINK) + suffix, 'exists:', Path(str(FOUNDER_PLINK) + suffix).exists())

    samples = read_table(SAMPLES)
    positions = read_table(POSITIONS)
    pedigree = read_table(PEDIGREE)

    required_samples = {'sample_id', 'bam_path'}
    required_positions = {'CHR', 'POS', 'REF', 'ALT'}
    missing_samples = required_samples - set(samples.columns)
    missing_positions = required_positions - set(positions.columns)
    if missing_samples:
        raise ValueError(f'samples table is missing {missing_samples}')
    if missing_positions:
        raise ValueError(f'positions table is missing {missing_positions}')

    sample_ids = samples['sample_id'].astype(str)
    if sample_ids.duplicated().any():
        raise ValueError('sample_id values must be unique')
    bad_bams = samples.loc[~samples['bam_path'].map(lambda p: Path(str(p)).exists()), ['sample_id', 'bam_path']]
    print('n_samples:', len(samples))
    print('n_positions:', len(positions))
    print('n_pedigree_rows:', len(pedigree))
    print('n_missing_bam_paths:', len(bad_bams))
    display(samples.head())
    display(positions.head())
    display(pedigree.head())
    if len(bad_bams):
        display(bad_bams.head(20))

if not DRY_RUN:
    validate_inputs()
else:
    print('DRY_RUN=True: input validation will run after you set real paths or DRY_RUN=False.')


## Chromosome Manifest

Edit this manifest to match your reference. For HS rats on rn8, autosomes are usually `chr1` to `chr20`; sex chromosomes and mitochondrial contigs can vary by reference (`chrX`, `chrY`, `chrM`, `MT`).

Ploidy policy used below:

| chromosome class | setting |
| --- | --- |
| autosomes | `--ploidy 2` |
| chrX | `--ploidy-males 1 --ploidy-females 2` |
| chrY | `--ploidy-males 1 --ploidy-females 0`; ploidy-zero samples stay in outputs as missing genotypes and zero haplotype probabilities |
| mitochondrial | usually `--ploidy 1`; adjust if your analysis requires otherwise |

Run STITCHV2 independently per chromosome, then aggregate pedigree QC across all successful first-pass chromosomes.

In [ ]:
autosomes = [f'chr{i}' for i in range(1, 21)]
chrom_rows = [{'chromosome': c, 'ploidy': 2, 'ploidy_males': None, 'ploidy_females': None, 'run': True} for c in autosomes]
chrom_rows += [
    {'chromosome': 'chrX', 'ploidy': None, 'ploidy_males': 1, 'ploidy_females': 2, 'run': True},
    {'chromosome': 'chrY', 'ploidy': None, 'ploidy_males': 1, 'ploidy_females': 0, 'run': True},
    {'chromosome': 'chrM', 'ploidy': 1, 'ploidy_males': None, 'ploidy_females': None, 'run': True},
]
chromosomes = pd.DataFrame(chrom_rows)
display(chromosomes)


## STITCHV2 Run Parameters Used Here

The command builder below intentionally uses STITCH parity settings for comparability:

- `--stitch-compat`: sets recombination rate to 0.5 cM/Mb, base and mapping quality filters to 17, caps base quality by mapping quality, limits insert size to 600, keeps ref/alt-only evidence, uses `fragment_likelihood_mode=replace`, and uses STITCH-like fragment likelihood caps.
- `--founder-plink ... --founder-immutable --n-founders 9`: loads 8 fixed founders and appends one mutable founder if the PLINK panel has 8 samples.
- `--em-iterations 40`: allows mutable founders to converge; adaptive EM can stop earlier.
- `--no-calibrate-genotype-posteriors --genotype-call-mode stitch_no_call`: no calibration, STITCH-like hard-call no-call gate.
- When `calibrate=True`, default standard callability uses `--calibration-callability-decision-mode per_snp_hierarchical`: one LightGBM callability model, per-SNP use/fallback decisions, and STITCH no-call fallback for unsafe SNPs. Set `GENOTYPE_CALL_MODE = 'quality_gated'` for the learned gate to affect hard calls.
- `--compact-evidence-cache-mode readwrite`: first pass builds cache; reruns can switch to `read`.
- `--executor dask --dask-processes` is intentionally not used. Local default is thread workers; use jobqueue only after a local-thread run is stable.

To see every CLI parameter from the installed package, run the help cell near the end of the notebook.

In [ ]:
def chromosome_args(row: pd.Series) -> list[str]:
    if pd.notna(row.get('ploidy_males')) and pd.notna(row.get('ploidy_females')):
        return ['--ploidy-males', str(int(row['ploidy_males'])), '--ploidy-females', str(int(row['ploidy_females']))]
    return ['--ploidy', str(int(row.get('ploidy', 2)))]

def stitchv2_run_cmd(
    row: pd.Series,
    *,
    output_dir: Path,
    cache_mode: str,
    pedigree: Path | None = None,
    pedigree_mode: str = 'off',
    pedigree_strength: float = 0.0,
    pedigree_iterations: int = 4,
    calibrate: bool = False,
) -> list[str]:
    chrom = str(row['chromosome'])
    cache_dir = CACHE_ROOT / chrom
    cmd = [
        STITCHV2, 'run',
        '--samples', str(SAMPLES),
        '--positions', str(POSITIONS),
        '--chromosome', chrom,
        '--output-dir', str(output_dir),
        '--founder-plink', str(FOUNDER_PLINK),
        '--founder-immutable',
        '--n-founders', str(N_FOUNDERS),
        '--stitch-compat',
        '--fragment-likelihood-mode', 'replace',
        '--em-iterations', str(EM_ITERATIONS),
        '--max-mem', str(MAX_MEM),
        '--hmm-backend', HMM_BACKEND,
        '--jax-persistent-cache-dir', str(JAX_CACHE),
        '--read-mode', 'read_stream',
        '--read-stream-backend', READ_BACKEND,
        '--compact-evidence-cache-dir', str(cache_dir),
        '--compact-evidence-cache-mode', cache_mode,
        '--compact-evidence-cache-format', 'parquet_zarr',
        '--compact-evidence-cache-sample-batch-size', '256',
        '--genotype-call-mode', GENOTYPE_CALL_MODE,
        '--write-genotype-posteriors',
        '--write-genotype-calls',
        '--write-haplotype-probabilities',
        '--write-support-mask',
        '--executor', EXECUTOR,
        '--dask-scheduler', DASK_SCHEDULER,
        '--dask-n-workers', str(DASK_N_WORKERS),
        '--dask-threads-per-worker', str(DASK_THREADS_PER_WORKER),
        '--dask-dashboard-address', DASK_DASHBOARD_ADDRESS,
        '--dask-performance-report', str(output_dir / 'dask_performance_report.html'),
        '--dask-task-stream', str(output_dir / 'dask_task_stream.html'),
        '--compression', 'zstd',
        '--compression-level', '6',
        '--random-seed', '1',
    ]
    if DASK_MEMORY_LIMIT:
        cmd += ['--dask-memory-limit', DASK_MEMORY_LIMIT]
    cmd += chromosome_args(row)
    if not calibrate:
        cmd += ['--no-calibrate-genotype-posteriors']
    else:
        cmd += ['--calibration-callability-decision-mode', CALIBRATION_CALLABILITY_DECISION_MODE]
    if pedigree is not None and pedigree_mode != 'off':
        cmd += [
            '--pedigree', str(pedigree),
            '--pedigree-mode', pedigree_mode,
            '--pedigree-strength', str(pedigree_strength),
            '--pedigree-iterations', str(pedigree_iterations),
            '--pedigree-kinship-threshold', '0.01',
        ]
    else:
        cmd += ['--pedigree-mode', 'off', '--pedigree-strength', '0']
    return cmd

def run_chromosome_pass(pass_name: str, *, cache_mode: str, pedigree: Path | None = None, pedigree_mode: str = 'off', pedigree_strength: float = 0.0, pedigree_iterations: int = 4) -> list[Path]:
    run_dirs = []
    for _, row in chromosomes.loc[chromosomes['run']].iterrows():
        chrom = str(row['chromosome'])
        out = RUN_ROOT / pass_name / chrom
        out.parent.mkdir(parents=True, exist_ok=True)
        cmd = stitchv2_run_cmd(
            row,
            output_dir=out,
            cache_mode=cache_mode,
            pedigree=pedigree,
            pedigree_mode=pedigree_mode,
            pedigree_strength=pedigree_strength,
            pedigree_iterations=pedigree_iterations,
        )
        run_checked(cmd)
        run_dirs.append(out)
    return run_dirs


## Pass 1: No Pedigree

This creates preliminary calls for pedigree QC. Use `readwrite` the first time to build compact evidence caches. If caches already exist and match the same samples, positions, and read filters, use `read` to skip BAM decoding.

In [ ]:
pass1_dirs = run_chromosome_pass(
    'pass1_no_pedigree_replace_k9_iter40',
    cache_mode='readwrite',
    pedigree=None,
    pedigree_mode='off',
)
pass1_dirs[:3], len(pass1_dirs)


## Global Pedigree QC

Use genotype calls from all first-pass chromosomes. `pedigree-qc` samples variants after per-variant missingness and MAF filters, computes sample-sample genotype correlation, checks declared parent-child edges, reports unexpected related pairs, writes a curated pedigree, and writes a UMAP before/after edge plot.

Main outputs:

- `pedigree_curated.parquet`: use this in second-pass STITCHV2 runs.
- `pedigree_edge_qc.parquet`: declared parent edges with observed relatedness and remove/retain status.
- `pedigree_sample_qc.parquet`: per-sample pass/review/corrected status.
- `sample_similarity.parquet`: high-similarity pairs and expected-vs-observed relationship annotations.
- `pedigree_long_edges.parquet`: long UMAP pedigree edges that deserve review.
- `pedigree_umap_before_after.html`: interactive hvplot/Bokeh UMAP with before/after pedigree edges.

In [ ]:
def pedigree_qc_cmd(run_dirs: list[Path]) -> list[str]:
    cmd = [
        STITCHV2, 'pedigree-qc',
        '--samples', str(SAMPLES),
        '--pedigree', str(PEDIGREE),
        '--output-dir', str(PEDIGREE_QC_DIR),
        '--sample-id-col', 'sample_id',
        '--pedigree-offspring-col', 'sample_id',
        '--pedigree-parent1-col', 'father_id',
        '--pedigree-parent2-col', 'mother_id',
        '--value-column', 'genotype_call',
        '--max-variants', '50000',
        '--min-call-rate', '0.80',
        '--min-maf', '0.005',
        '--report-min-r', '0.59',
        '--unrelated-max-r', '0.59',
        '--first-degree-min-r', '0.64',
        '--same-min-r', '0.88',
        '--unlink-calls', 'unrelated,same',
        '--sample-block-size', '1024',
        '--max-full-matrix-samples', '5000',
        '--umap-neighbors', '50',
        '--umap-max-variants', '10000',
        '--random-seed', '1',
    ]
    for d in run_dirs:
        cmd += ['--run-output-dir', str(d)]
    return cmd

run_checked(pedigree_qc_cmd(pass1_dirs))


## Review Pedigree QC

Do not blindly use the curated pedigree. Review removed parent edges, inconclusive edges, unexpected related pairs, and long UMAP edges. If many true parent-child edges are being removed, tune the relatedness thresholds before rerunning `pedigree-qc`.

In [ ]:
if PEDIGREE_QC_DIR.exists():
    summary_path = PEDIGREE_QC_DIR / 'pedigree_qc_summary.json'
    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        print(json.dumps(summary, indent=2)[:4000])
    for name in ['pedigree_edge_qc.parquet', 'pedigree_sample_qc.parquet', 'sample_similarity.parquet', 'pedigree_long_edges.parquet']:
        path = PEDIGREE_QC_DIR / name
        if path.exists():
            print('\n', name)
            display(pd.read_parquet(path).head(20))
    html = PEDIGREE_QC_DIR / 'pedigree_umap_before_after.html'
    if html.exists():
        from IPython.display import IFrame, display
        display(IFrame(str(html), width=1400, height=750))
else:
    print('Pedigree QC output does not exist yet.')


## Pass 2: Rerun With Curated Pedigree

Recommended starting mode is `transmission` because it uses parent-child Mendelian message passing and can also let offspring/siblings inform parents. `kinship` is a sparse fallback from the pedigree graph. `smooth` is cheapest and most conservative because it only blends dosage toward parent means.

Start with one chromosome and compare diagnostics before launching all chromosomes. If the curated pedigree removes many edges, inspect those samples before trusting pedigree-adjusted calls.

In [ ]:
CURATED_PEDIGREE = PEDIGREE_QC_DIR / 'pedigree_curated.parquet'

# Choose one of: 'transmission', 'kinship', 'smooth'.
PEDIGREE_MODE = 'transmission'
PEDIGREE_STRENGTH = 0.70
PEDIGREE_ITERATIONS = 6

pass2_dirs = run_chromosome_pass(
    f'pass2_curated_pedigree_{PEDIGREE_MODE}',
    cache_mode='read',
    pedigree=CURATED_PEDIGREE,
    pedigree_mode=PEDIGREE_MODE,
    pedigree_strength=PEDIGREE_STRENGTH,
    pedigree_iterations=PEDIGREE_ITERATIONS,
)
pass2_dirs[:3], len(pass2_dirs)


## Combine Per-Chromosome Outputs

STITCHV2 writes partitioned Parquet by default. Keep Parquet/Zarr as the primary storage. Export BCF only for downstream tools that require it; BCF export is an interoperability path and will slow down I/O.

In [ ]:
def combine_cmd(run_dir: Path, output_dir: Path) -> list[str]:
    return [
        STITCHV2, 'combine',
        '--run-output-dir', str(run_dir),
        '--datasets', 'all',
        '--output-dir', str(output_dir),
        '--write-zarr',
        '--zarr-output', str(output_dir / 'float_outputs.zarr'),
        '--zarr-consolidated',
        '--compression', 'zstd',
        '--compression-level', '6',
    ]

for d in pass2_dirs:
    run_checked(combine_cmd(d, d / 'combined'))


## Whole-Genome Run Summaries

Each chromosome run writes `run_summary.json`, `stage_timings.json`, `diagnostics_summary.json`, and optionally `dask_run_summary.json`. This cell collects runtime, memory, cache, EM, calibration, diagnostics, and pedigree status across chromosomes.

In [ ]:
def load_json(path: Path) -> dict:
    return json.loads(path.read_text()) if path.exists() else {}

def summarize_runs(run_dirs: list[Path]) -> pd.DataFrame:
    rows = []
    for d in run_dirs:
        timing_path = d / 'stage_timings.json'
        timings = json.loads(timing_path.read_text()) if timing_path.exists() else []
        diag = load_json(d / 'diagnostics_summary.json')
        summary = load_json(d / 'run_summary.json')
        rows.append({
            'chromosome': d.name,
            'elapsed_seconds': summary.get('elapsed_seconds'),
            'peak_rss_mb': summary.get('peak_rss_mb'),
            'read_s': sum(float(x.get('seconds_read_extract', 0)) for x in timings),
            'hmm_s': sum(float(x.get('seconds_hmm', 0)) for x in timings),
            'calibration_s': sum(float(x.get('seconds_calibration', 0)) for x in timings),
            'write_s': sum(float(x.get('seconds_write', 0)) for x in timings),
            'cache_hit_blocks': sum(bool(x.get('compact_cache_hit')) for x in timings),
            'em_updates': sum(int(x.get('em_updates', 0)) for x in timings),
            'diagnostics_status': diag.get('status'),
            'diagnostics_warnings': ','.join(diag.get('warnings', [])) if isinstance(diag.get('warnings'), list) else '',
            'diagnostics_failures': ','.join(diag.get('failures', [])) if isinstance(diag.get('failures'), list) else '',
            'pedigree_mode': summary.get('pedigree', {}).get('mode') if isinstance(summary.get('pedigree'), dict) else None,
        })
    return pd.DataFrame(rows)

if 'pass2_dirs' in globals():
    display(summarize_runs(pass2_dirs))
elif 'pass1_dirs' in globals():
    display(summarize_runs(pass1_dirs))


## Parameter Reference

The cells below print the exact parser for the installed package. Treat this as the source of truth for all parameters in the environment that will run the full genome.

In [ ]:
print(capture([STITCHV2, 'run', '--help']))


In [ ]:
print(capture([STITCHV2, 'pedigree-qc', '--help']))


## Interactive Python API Skeleton

The CLI is preferred for whole-genome production runs. For interactive experiments, the same settings can be passed with `PipelineConfig`.

In [ ]:
from stitchv2 import PipelineConfig, StitchPipeline

example_config = PipelineConfig(
    chromosome='chr12',
    positions_path=POSITIONS,
    output_dir=RUN_ROOT / 'interactive_chr12_example',
    n_founders=N_FOUNDERS,
    max_mem=MAX_MEM,
    em_iterations=EM_ITERATIONS,
    hmm_backend='jax',
    read_mode='read_stream',
    read_stream_backend='stitch_style_bamreader',
    fragment_likelihood_mode='replace',
    fragment_coupling_model='stitch_parity',
    calibrate_genotype_posteriors=False,
    genotype_call_mode='stitch_no_call',
    write_genotype_posteriors=True,
    write_genotype_calls=True,
    write_haplotype_probabilities=True,
    compact_evidence_cache_dir=CACHE_ROOT / 'chr12',
    compact_evidence_cache_mode='readwrite',
    pedigree_mode='off',
)
example_config

# To run interactively, load samples and founders explicitly or use the CLI for PLINK founder loading.
# samples = read_table(SAMPLES)
# StitchPipeline(example_config).prepare_inputs(samples)
